Loading data and isolate the target

In [14]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import joblib

ROOT = Path.cwd().parent
raw_path = ROOT / "data" / "raw" / "raw_data.csv"
df = pd.read_csv(raw_path).drop(columns=["Student_ID"])

X = df.drop(columns="Mental_Health_Risk")
y = df['Mental_Health_Risk']

Spliting the Data

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state= 42, stratify=y)

Group columns by preprocessing

In [3]:
numeric_features = [
    'Age',
    'Year_of_Study',
    'Sleep_Hours',
    'Study_Hours_Per_Week',
    'GPA',
    'Social_Media_Hours',
    'Screen_Time_Hours',
    'Financial_Stress',
    'Family_Support',
    'Attendance_Rate',
]

ordinal_features = ['Physical_Activity', 'Diet_Quality']
ordinal_categories = [['Low', 'Moderate', 'High'], ['Poor', 'Average', 'Good']]

nominal_features = [
    'Gender',
    'Part_Time_Job',
    'Previous_Mental_Health_History',
]

Build ColumnTransformer pipeline

In [4]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=ordinal_categories)),
    ]
)

nominal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        (
            'encoder',
            OneHotEncoder(
                drop='first', sparse_output=False, handle_unknown='ignore'
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features),
    ]
)

Fit on X_train and transform both splits

In [5]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

Convert back to DataFrames with clean column names

In [10]:
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_test_df = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)

print(
    f'Train shape: {X_train_df.shape} | Nulls: {X_train_df.isnull().sum().sum()}'
)
print(
    f'Test shape:  {X_test_df.shape}  | Nulls: {X_test_df.isnull().sum().sum()}'
)

Train shape: (1600, 16) | Nulls: 0
Test shape:  (400, 16)  | Nulls: 0


Save the fitted preprocessor

In [15]:
models_dir = ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

preprocessor_path = models_dir / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

['C:\\Users\\lioub\\PycharmProjects\\MindPulse-AI\\models\\preprocessor.joblib']